## GENeSYS-MOD RES Tool - EUROPE hub (batch/tiling workflow)

Continent-scale run at country level over the full official Europe hub (Europe +
Turkey/Caucasus + Israel + Central Asia). A single cutout over the whole extent (plus
rasterizing the land-cover layer over it) exhausts a laptop's RAM, so this driver
**tiles** the area into disjoint country groups, builds one small ERA5 cutout per
tile, runs the full pipeline (capacity factors + GIS potentials) on that tile, frees
the cutout, then **combines everything into single outputs** (one CSV per
tech/category, one combined potentials table, one combined map). Peak RAM is bounded
by the largest single tile, not the whole extent.

All the batch logic is `run_batch_by_tile` in `functions.py`; this notebook only sets
paths/parameters and the editable `TILES` dict. Region-geojson workflow (a literal
`region` column, no `iso_a2`).

Data layers:
- **regions**: `geodata/regions/europe_regions.geojson` (51 hub regions, `region` = ISO-A2),
  derived from the user's `res_countries_europe_mainland.geojson` by
  `build_europe_regions.py` (drops `iso_a2` so the region workflow fires, dissolves to
  one geometry per region, clips Svalbard/Jan Mayen off Norway). The three microstates
  (LI, MC, SM) are in the file but not assigned to a tile below - see the TILES cell.
- **land cover / protected areas**: TODO - a Europe land-cover raster (CORINE or ESA
  WorldCover) + WDPA protected areas. Until those exist, set `run_potentials=False`
  below and the batch still produces all the capacity-factor timeseries. See the
  exclusion-layers cell.
- **offshore**: optional EEZ + bathymetry (leave `offshore_file=None` to skip;
  landlocked tiles are handled cleanly either way).

In [1]:
from functions import *

In [2]:
onshore_turbine = 'Vestas_V112_3MW'
offshore_turbine = 'Vestas_V164_7MW_offshore'
solar_panel = 'CSi'

# Europe hub geometry (51 nodes, region = ISO-A2). Built by build_europe_regions.py
# from res_countries_europe_mainland.geojson. The 'region' column (and NO 'iso_a2')
# triggers the region-geojson workflow the batch process needs.
geo_file = "geodata/regions/europe_regions.geojson"

# Offshore. One whole-hub EEZ polygon (built by build_eez_europe.py from eez_v12.gpkg
# by ISO_SOV1, with the Caspian-only sovereigns AZ/KZ/TM dropped and geometries clipped
# to a Europe bbox). _get_coords_custom clips it to each tile's cutout bbox, so one file
# serves every tile; landlocked tiles simply find no sea. Bathymetry is PER TILE
# (geodata/offshore/bathymetry_europe_{tile}.nc, built by build_bathymetry_europe.py) and wired
# through the TILE_LAYERS dict below, so the global bathymetry_file stays None.
offshore_file = "geodata/offshore/eez_europe.geojson"
bathymetry_file = None

# 'Prove fast, scale later': one day first to smoke-test the whole tiling machinery;
# switch to "2018" for a full year (cutouts + ERA5 chunk cache are reused).
timeframe = "2012"
filename = "europe"          # BASE prefix; per-tile calls use f"{filename}_{tile}"
output_dir = create_output_folder(timeframe)

admin = 0                    # country level

# Availability-map colour: a hex ('#004664' -> white->navy ramp) or a named
# matplotlib colormap ('Greens'). None = default Greens.
map_color = '#004664'

The output folder already exists.


In [3]:
# Cutout resolution (degrees). Coarse (~0.275 deg ~= 30 km) for the dry run; ERA5 is
# ~0.25-0.3 deg native anyway, so this is a sensible continent-scale default.
dx_step = 0.275
dy_step = 0.275

# Buffer (degrees) added around each tile's country polygons when sizing its cutout,
# so coastal cells / border cells are not clipped. Overlaps between neighbouring
# tiles' buffers are fine - the shared ERA5 chunk cache reuses the downloads.
bbox_buffer = 1.0

# PV slope/azimuth for utility-scale (ground-mounted) PV.
pv_slope = 36.7
pv_azimuth = 180
pv_optimal_tilt = True
# Rooftop PV, split like the model's P_PV_Rooftop_Residential / _Commercial. Each
# segment gets its own CF timeseries from a mix of panel orientations (tilt in
# degrees or "optimal"; azimuth "equator" = facing the equator per cell, "east",
# "west", or degrees from north) and a share of the rooftop potential.
# Residential roofs face wherever the house does; commercial roofs are mostly flat,
# so racks can face the equator at the optimal tilt.
rooftop_segments = {
    "residential": [{"tilt": 20, "azimuth": "equator", "weight": 0.5},
                    {"tilt": 20, "azimuth": "east", "weight": 0.25},
                    {"tilt": 20, "azimuth": "west", "weight": 0.25}],
    "commercial": [{"tilt": "optimal", "azimuth": "equator", "weight": 1.0}],
}
# Share of the rooftop potential per segment, per country, from GHSL built-up
# surface (geodata/config/rooftop_shares.csv, built by build_rooftop_shares.py).
rooftop_shares = load_rooftop_shares(geo_file=geo_file)   # region codes are ISO2

# Usable-site thresholds (min developable share of a cell to count as a usable site).
usable_threshold = 0.01
usable_threshold_rooftop = 0.05

## TILES - the one thing you edit

Group the countries into tiles (country groups). Each tile becomes one small cutout.
**Every region must appear in exactly one tile** (disjoint - `run_batch_by_tile`
asserts this to prevent double-counting). Keep tiles geographically compact so each
cutout stays small; a tile with more, spread-out countries makes a bigger cutout.

The default below covers all 45 regions in `europe_regions.geojson`. Edit freely -
merge tiles for fewer/bigger cutouts, split for smaller ones.

In [4]:
# Full official Europe hub: 51 regions incl. Israel and Central Asia (IL, KZ, UZ, TM,
# TJ, KG). They sit far east/south of Europe proper, so they get their own tiles
# (central_asia) or join the nearest one (IL -> turkey_levant); central_asia is 12.6k
# cells at 0.275 deg, still smaller than the nordics tile, so it stays RAM-safe.
#
# Microstates LI, MC, SM are in europe_regions.geojson but intentionally left out of
# any tile: each is smaller than one 0.275 deg cutout cell (~900 km^2), so no grid-cell
# centre lands inside them and they get no timeseries (true of a single-cutout run at
# this resolution too, not a batch artefact). AD (Andorra) is kept - its bbox spans
# >0.275 deg so it straddles a cell boundary and catches one. To include a microstate,
# either merge it into its neighbour in europe_regions.geojson or use finer dx/dy.
TILES = {
    "iberia":        ["ES", "PT", "AD"],
    "france_bnl":    ["FR", "BE", "NL", "LU"],
    "british_isles": ["GB", "IE"],
    "central":       ["DE", "AT", "CH", "CZ", "PL", "SK", "HU"],
    "nordics":       ["SE", "NO", "FI", "DK"],
    "italy_alps":    ["IT", "SI", "HR"],
    "balkans":       ["GR", "BG", "RO", "RS", "MK", "AL", "ME", "BA", "XK"],
    "baltics":       ["EE", "LV", "LT"],
    "east":          ["UA", "MD", "BY"],
    "turkey_levant": ["TR", "GE", "AM", "AZ", "IL"],
    "central_asia":  ["KZ", "UZ", "TM", "TJ", "KG"],
}

## Exclusion layers (land cover + protected areas)

Region-agnostic builders; only the paths/codes are region-specific. **These layers do
not exist for Europe yet** (CORINE/WorldCover + WDPA are the intended sources). Two
modes:

- **Timeseries-only (default here)**: leave `LANDCOVER = None`; `run_potentials` below
  becomes `False` and the batch produces all capacity-factor timeseries (PV, onshore
  wind, offshore wind if enabled, heat) without GIS potentials. This is the
  'prove fast' path.
- **Full potentials**: set `LANDCOVER` to a Europe land-cover raster, fill the exclude
  codes + WDPA query (mirror `GENeSYS-MOD_RES_Tool_IN.ipynb`), and `run_potentials`
  turns on. The excluder `crs` must be a PROJECTED metre CRS (equal_area_crs) or land
  availability comes back ZERO everywhere (CLAUDE.md projected-CRS gotcha).

In [5]:
# === Land-cover + protected-area layers for GIS potentials (per-tile) ===
# Europe is covered by two land-cover rasters with DIFFERENT class schemes and CRS, so
# each tile names its own raster + codes through the TILE_LAYERS dict below; functions.py
# builds that tile's excluders lazily and frees them with the cutout. No region logic
# lives in functions.py - only these generic per-tile specs.
#
# 1) CORINE (geodata/landcover/corine.tif) covers the 9 European tiles. IMPORTANT: this file uses
#    the REMAPPED CLC GRID_CODE 1..44 (verified live with np.unique - NOT the raw CLC
#    111..523 codes), int8, nodata -128, EPSG:3035 (metres). Because it is already in a
#    metre CRS, raster_crs=None uses the native 3035 and satisfies the projected-CRS
#    gotcha with no reprojection. Strict-exclude GRID_CODEs: artificial surfaces 1-11,
#    permanent snow 34, wetlands 35-39, water bodies 40-44. Keep forest/shrub/grass/bare/
#    agriculture (12-33) developable. Rooftop proxy keeps the urban fabric 1-2.
CORINE = "geodata/landcover/corine.tif"
CORINE_EXCLUDE = list(range(1, 12)) + [34] + list(range(35, 45))   # 1-11, 34, 35-44
CORINE_ROOFTOP = [1, 2]                                            # continuous+discont. urban

# 2) ESA WorldCover fill (geodata/landcover/worldcover_europe_east_300m.tif) covers the two tiles
#    CORINE does not reach: turkey_levant and central_asia. It is EPSG:4326 (degrees), so
#    its excluder MUST use a projected CRS - the "__equal_area__" sentinel makes
#    functions.py use equal_area_crs(tile_shapes), else land availability comes back ZERO
#    everywhere (CLAUDE.md projected-CRS gotcha). Classes: 10 tree, 20 shrub, 30 grass,
#    40 crop, 50 built, 60 bare, 70 snow, 80 water, 90 wetland, 95 mangrove, 100 moss,
#    0 fill. Strict-exclude 0/built/water/snow/wetland/mangrove; keep tree/shrub/grass/
#    bare/crop developable. Rooftop keeps built-up 50.
WC_EAST = "geodata/landcover/worldcover_europe_east_300m.tif"
WC_EXCLUDE = [0, 50, 70, 80, 90, 95]
WC_ROOFTOP = [50]

# Protected areas - one merged, strict-tier WDPA GeoPackage for the whole hub (built by
# build_wdpa_europe.py, filtered to IUCN Ia/Ib/II at build time). functions.py' vector
# cache reads+filters it once and reuses it across every tile. Terrestrial+coastal strict
# areas exclude land; marine+coastal strict areas exclude offshore.
WDPA = "geodata/exclusions/wdpa_europe.gpkg"
WDPA_LAYER = "wdpa"
PROTECTED_QUERY = "REALM in ['Terrestrial', 'Coastal']"      # WDPA already IUCN Ia/Ib/II
OFFSHORE_QUERY  = "REALM in ['Marine', 'Coastal']"           # (build-time IUCN filter)


def corine_tile(bathymetry=None, offshore_file="__default__"):
    """A TILE_LAYERS spec for a CORINE-covered tile. bathymetry -> per-tile depth grid;
    offshore_file left at the sentinel uses the global EEZ (pass None to force landlocked)."""
    spec = {
        "land_raster": CORINE,
        "land_exclude_codes": CORINE_EXCLUDE,
        "rooftop_codes": CORINE_ROOFTOP,
        "raster_crs": None,                       # native EPSG:3035 (metres) - no gotcha
        "protected_files": WDPA,
        "protected_layer": WDPA_LAYER,
        "protected_query": PROTECTED_QUERY,
        "offshore_exclude_files": WDPA,
        "offshore_exclude_layer": WDPA_LAYER,
        "offshore_exclude_query": OFFSHORE_QUERY,
    }
    if bathymetry is not None:
        spec["bathymetry_file"] = bathymetry
    if offshore_file != "__default__":
        spec["offshore_file"] = offshore_file
    return spec


def worldcover_east_tile(bathymetry=None, offshore_file="__default__"):
    """A TILE_LAYERS spec for a WorldCover-fill tile (turkey_levant / central_asia).
    raster_crs='__equal_area__' is mandatory (EPSG:4326 raster -> projected-CRS gotcha)."""
    spec = {
        "land_raster": WC_EAST,
        "land_exclude_codes": WC_EXCLUDE,
        "rooftop_codes": WC_ROOFTOP,
        "raster_crs": "__equal_area__",           # geographic raster -> per-tile LAEA
        "protected_files": WDPA,
        "protected_layer": WDPA_LAYER,
        "protected_query": PROTECTED_QUERY,
        "offshore_exclude_files": WDPA,
        "offshore_exclude_layer": WDPA_LAYER,
        "offshore_exclude_query": OFFSHORE_QUERY,
    }
    if bathymetry is not None:
        spec["bathymetry_file"] = bathymetry
    if offshore_file != "__default__":
        spec["offshore_file"] = offshore_file
    return spec


def bathy(tile):
    return f"geodata/offshore/bathymetry_europe_{tile}.nc"


# Per-tile layer map. Nine CORINE tiles (each coastal one gets its own GMRT bathymetry;
# 'central'/'east' are coastal too - Baltic/North Sea and Black Sea). Two WorldCover-fill
# tiles: turkey_levant coastal (Black Sea + Med), central_asia landlocked (offshore_file
# =None so no sea is assigned even though the global EEZ is set). Keys must match TILES.
TILE_LAYERS = {
    "iberia":        corine_tile(bathymetry=bathy("iberia")),
    "france_bnl":    corine_tile(bathymetry=bathy("france_bnl")),
    "british_isles": corine_tile(bathymetry=bathy("british_isles")),
    "central":       corine_tile(bathymetry=bathy("central")),
    "nordics":       corine_tile(bathymetry=bathy("nordics")),
    "italy_alps":    corine_tile(bathymetry=bathy("italy_alps")),
    "balkans":       corine_tile(bathymetry=bathy("balkans")),
    "baltics":       corine_tile(bathymetry=bathy("baltics")),
    "east":          corine_tile(bathymetry=bathy("east")),
    "turkey_levant": worldcover_east_tile(bathymetry=bathy("turkey_levant")),
    "central_asia":  worldcover_east_tile(offshore_file=None),   # landlocked
}

# Global excluders stay None: every tile supplies its own via TILE_LAYERS, so the
# global ones are never used. run_potentials on -> GIS potentials run per tile.
excluder = cities = wind_excluder = None
run_potentials = True
print(f"run_potentials = {run_potentials}; {len(TILE_LAYERS)} tiles have layer specs")
print("  CORINE tiles :", [t for t, s in TILE_LAYERS.items() if s['land_raster'] == CORINE])
print("  WorldCover   :", [t for t, s in TILE_LAYERS.items() if s['land_raster'] == WC_EAST])
print("  landlocked   :", [t for t, s in TILE_LAYERS.items() if s.get('offshore_file', 1) is None])

run_potentials = True; 11 tiles have layer specs
  CORINE tiles : ['iberia', 'france_bnl', 'british_isles', 'central', 'nordics', 'italy_alps', 'balkans', 'baltics', 'east']
  WorldCover   : ['turkey_levant', 'central_asia']
  landlocked   : ['central_asia']


## Estimate first (no download)

Print each tile's cutout grid size + the total and the peak single tile (which bounds
peak RAM). Adjust `TILES` / `dx_step` / `dy_step` if a tile is too big before
downloading anything.

In [6]:
_ = run_batch_by_tile(
    TILES, geo_file, timeframe, filename,
    dx=dx_step, dy=dy_step, bbox_buffer=bbox_buffer,
    estimate_only=True)

Note: 3 region(s) in the geojson are in no tile: ['LI', 'MC', 'SM']

=== Tile 'iberia' -> ['ES', 'PT', 'AD'] (filename=europe_iberia) ===
Expected cutout size:
  bounds (lon/lat) : [-32.28, 5.34] x [26.64, 44.79]
  grid             : 136 x 66 = 8,976 cells  (dx=0.275, dy=0.275)
  time steps       : 8,784 hourly  (2012)
  points/variable  : 78,845,184
  total points     : 867,297,024  (x11 variables)
  uncompressed     : ~3.47 GB float32 (on-disk .nc is far smaller, compressed)

=== Tile 'france_bnl' -> ['FR', 'BE', 'NL', 'LU'] (filename=europe_france_bnl) ===
Expected cutout size:
  bounds (lon/lat) : [-6.13, 10.56] x [40.37, 54.56]
  grid             : 60 x 51 = 3,060 cells  (dx=0.275, dy=0.275)
  time steps       : 8,784 hourly  (2012)
  points/variable  : 26,879,040
  total points     : 295,669,440  (x11 variables)
  uncompressed     : ~1.18 GB float32 (on-disk .nc is far smaller, compressed)

=== Tile 'british_isles' -> ['GB', 'IE'] (filename=europe_british_isles) ===
Expected cuto

## Run the batch

One call does everything: per tile it estimates, builds/reuses the cutout, assigns
coords, runs the capacity factors and (if `run_potentials`) the GIS potentials, then
frees the cutout. After the loop it writes the combined CSVs under the base
`filename` and returns the combined dataframes + per-tile maps.

The first run downloads ERA5 and will take a while. Re-running reuses each tile's
cutout `.nc` and the ERA5 chunk cache; set `skip_if_combined_exists=True` to also skip
tiles whose combined CSVs already exist.

In [7]:
result = run_batch_by_tile(
    TILES, geo_file, timeframe, filename,
    dx=dx_step, dy=dy_step, bbox_buffer=bbox_buffer,
    offshore_file=offshore_file, bathymetry_file=bathymetry_file, admin=admin,
    solar_panel=solar_panel, onshore_turbine=onshore_turbine, offshore_turbine=offshore_turbine,
    pv_slope=pv_slope, pv_azimuth=pv_azimuth, optimal_tilt=pv_optimal_tilt,
    rooftop_segments=rooftop_segments, rooftop_shares=rooftop_shares,
    excluder=excluder, cities=cities,
    offshore_exclude_files=OFFSHORE_EXCLUDE_FILES, offshore_exclude_layer=OFFSHORE_EXCLUDE_LAYER,
    offshore_exclude_query=OFFSHORE_EXCLUDE_QUERY,
    usable_threshold=usable_threshold, usable_threshold_rooftop=usable_threshold_rooftop,
    run_pv=True, run_wind_onshore=True, run_wind_offshore=(offshore_file is not None),
    run_temperature=True, run_potentials=run_potentials,
    skip_if_combined_exists=False,
    output_dir=output_dir, color=map_color)

print("tiles run    :", result["tiles_run"])
print("tiles skipped:", result["tiles_skipped"])
print("missing codes:", result["missing_codes"])
print("combined CF families:", sorted(result["cf"].keys()))

NameError: name 'OFFSHORE_EXCLUDE_FILES' is not defined

## Combined outputs

Combined CSVs are on disk under the base `filename` (e.g.
`{timeframe}_{filename}_pv_opt.csv`, `{filename}_potentials_combined.csv`). Below: a
quick sanity check that a combined CF frame carries every region's column, and the
combined availability map across all tiles.

In [8]:
# Sanity check: the combined PV-opt frame should have one column per region across ALL tiles.
if "pv_opt" in result["cf"]:
    pv_opt = result["cf"]["pv_opt"]
    all_codes = sorted({c for codes in TILES.values() for c in codes})
    print(f"pv_opt: {pv_opt.shape[1]} region columns, {pv_opt.shape[0]} timesteps")
    print("missing region columns:", sorted(set(all_codes) - set(pv_opt.columns)))

if result["potentials_combined"] is not None:
    display(result["potentials_combined"])

pv_opt: 42 region columns, 8760 timesteps
missing region columns: ['LI', 'MC', 'SM']


In [9]:
# Combined availability map across all tiles (only if GIS potentials ran).
if result["maps"]:
    combine_and_plot_tile_maps(result["maps"], which="land", geo_file=geo_file, color=map_color)
else:
    print("No maps (run_potentials was False - set a LANDCOVER raster to enable).")

No maps (run_potentials was False - set a LANDCOVER raster to enable).


## Scale up to a full year

Once the dry run looks right, set `timeframe = "2018"` in the config cell and re-run.
The dry-run day's cutouts and the ERA5 chunk cache are reused; the full year downloads
the remaining hours into the same cache. Everything else is unchanged.